## Environment Setup

**Two conda environments are used in this notebook:**

- **`compass-hairpin`** — the default/primary environment. Everything in this notebook runs here *except* the two cells explicitly marked below.
- **`deeplabcut`** — only needed for the "DeepLabCut Import" cell and the "Analyze the Videos with DeepLabCut" cell, i.e. only when DLC tracking itself still needs to be run for one or more sessions. This project's `deeplabcut` env has DLC 2.3.11 + TensorFlow 2.10.1 on Python 3.10.18 (matched to whatever model/config you're analyzing with — verify against your own trained model's requirements if reusing this notebook for a different one).

DeepLabCut is a markerless pose estimation toolkit that uses deep learning to track user-defined body parts (Mathis et al., 2018).

### Installation Instructions

For detailed installation instructions, see the [DeepLabCut installation guide](https://deeplabcut.github.io/DeepLabCut/docs/installation.html).

---

### Reference

**Mathis, A., Mamidanna, P., Cury, K.M. et al.** (2018). DeepLabCut: markerless pose estimation of user-defined body parts with deep learning. *Nature Neuroscience*, 21, 1281–1289. https://doi.org/10.1038/s41593-018-0209-y

# Install Necessary Packages (Only need to do once)

In [ ]:
# autoreload
%load_ext autoreload
%autoreload 2

# Imports

In [ ]:
# Importing the toolbox (takes several seconds)
import pandas as pd
import numpy as np
from pathlib import Path
from pylab import *
import os
import shutil
import geopandas as gpd
import scipy.stats as sp
from datetime import datetime
import sys

from pathlib import Path

# Specify path to dlc_utils file
custom_utils_path = Path(os.getcwd()).parent/'src'/'compass_labyrinth'/'behavior'/'pose_estimation'
sys.path.append(str(custom_utils_path))

import dlc_utils

# Imports for DLC

**Needed now** — 5 sessions still don't have DLC output, so this cell and the "Analyze the Videos with DeepLabCut" section further down need to actually run for those. This requires switching the notebook's kernel to the `deeplabcut` conda env for this cell and the analysis cell specifically — everything else in this notebook runs fine in `compass-hairpin`. Fill in `dlc_config_path` above with the real path to the trained model's `config.yaml` first.

In [ ]:
# DeepLabCut Import
import deeplabcut
import tensorflow as tf
import keras
os.environ['DLClight'] = 'True'
print(tf.__version__)

### Specify Paths and Other Information

In [ ]:
##### CONFIGURE PATHS FOR YOUR LOCAL SYSTEM #####

# === VIDEO FILE LOCATIONS ===
# Original combined videos (4 mice per video, cropped per-mouse downstream).
video_path_1 = r'PATH\TO\PROJECT\Media Files'
video_path_2 = ''

# Project folder: renamed videos + reorganized DLC output + everything this
# notebook produces all live here, using the SessionXXXX convention (matching
# what DLC would have produced had it run on the renamed videos directly).
source_data_path = r'PATH\TO\PROJECT\compass_analysis'
videofile_path = os.path.join(source_data_path, 'videos_original')

# === DEEPLABCUT CONFIGURATION ===
# Needed only for the "Analyze the Videos with DeepLabCut" cell, to run DLC
# on the 5 sessions that haven't been analyzed yet. Requires the `deeplabcut`
# conda env/kernel for that cell specifically -- everything else in this
# notebook runs fine in `compass-hairpin`.
dlc_config_path = Path(r'PATH\TO\DLC_PROJECT\config.yaml')
dlc_scorer = 'YOUR_DLC_SCORER'

# === OUTPUT PATHS ===
frames_path = os.path.join(source_data_path, 'frames')  # only needed for click-based boundary selection
grid_boundaries_path = os.path.join(source_data_path, 'grid_boundaries')
dlc_cropping_bounds_path = os.path.join(source_data_path, 'dlc_cropping_bounds')
dlc_results_path = os.path.join(source_data_path, 'dlc_results')
grid_path = os.path.join(source_data_path, 'grid_files')
figure_path = os.path.join(source_data_path, 'figures')

# === METADATA ===
user_metadata_file_path = r'PATH\TO\PROJECT\metadata.xlsx'
trial_type = 'YOUR_TRIAL_TYPE'

# === SUMMARY ===
print("=== Path Configuration ===")
print(f"Video Source 1: {video_path_1}")
print(f"Video Source 2: {video_path_2}")
print(f"Central Video Location: {videofile_path}")
print(f"Metadata: {user_metadata_file_path}")
print(f"Figures Output: {figure_path}")

# Import Mouse Metadata

In [ ]:
# Add Metadata to metadata directory
mouseinfo = dlc_utils.import_cohort_metadata(user_metadata_file_path, trial_type)

# Validate the metadata
dlc_utils.validate_metadata(mouseinfo)

# Display summary of the metadata
dlc_utils.display_metadata_summary(mouseinfo)

mouseinfo.head()

# OPTIONAL: Copy and rename videos from original location to VIDEOFILE_PATH


**Run this** (not skipped anymore) — copies each session's video from `Media Files\` and renames it to `SessionXXXX.mp4` in `videos_original\`, so it lines up with the DLC output naming below. The default `source_naming="trial"` already matches this data's `"Trial     N.mp4"` filenames (verified against the actual `copy_and_rename_videos()` code), so this cell works unmodified.

In [ ]:
now = datetime.now(); print(now)

# 1. Save first frames
copy_results = dlc_utils.copy_and_rename_videos(
    mouseinfo_df=mouseinfo,
    video_paths=[video_path_1],
    destination_path=videofile_path,
    source_naming="trial",   # "trial" | "session-dash" | "session" | "format"
    source_format=None  # used when source_naming="format", e.g. "Session-{}.mp4"
)

# Get DeepLabCut Cropping Bounds

In [ ]:
# Save first frames for all videos (needed so there's something to click on)
now = datetime.now(); print(now)

print("\nSaving first frames for all videos...")
frame_results = dlc_utils.batch_save_first_frames(
    mouseinfo_df=mouseinfo,
    video_directory=videofile_path,
    frames_directory=frames_path
)

In [ ]:
# Get the DLC cropping bounds for all videos (interactive -- click 4 corners per session)
now = datetime.now(); print(now)

coordinates_dict = dlc_utils.batch_get_boundary_and_cropping(
    mouseinfo_df=mouseinfo,
    frames_directory=frames_path,
    cropping_directory=dlc_cropping_bounds_path,
    boundaries_directory=grid_boundaries_path,
    reprocess_existing=False  # set to True if you'd like to redo boundaries
)

In [ ]:
# Alternative: get cropping bounds for a single session (e.g. to redo just one)
coordinates_dict = dlc_utils.get_labyrinth_boundary_and_cropping(
    frames_directory=frames_path,
    cropping_directory=dlc_cropping_bounds_path,
    boundaries_directory=grid_boundaries_path,
    session='Session0001',  # change to the session you want to (re)select
    chamber_info=None
)

# Analyze the Videos with DeepLabCut

In [ ]:

# Prepare sessions -- automatically skips any session that already has DLC
# output (the 16 already-copied sessions), only queues the ones missing it
sessions_to_analyze, prep_summary = dlc_utils.prepare_dlc_analysis(
    mouseinfo,
    videofile_path,
    dlc_cropping_bounds_path,
    dlc_results_path,
)

# Print preparation summary
print(f"\nPreparation Summary:")
print(f"Ready for analysis: {prep_summary['ready_for_analysis']}")
print(f"Already analyzed: {prep_summary['skipped_existing']}")
print(f"Missing videos: {prep_summary['missing_video']}")
print(f"Missing coordinates: {prep_summary['missing_coordinates']}")

# Now run DLC analysis in the correct environment
analysis_results = []
for session in sessions_to_analyze:
    print(f"\nAnalyzing {session['session_name']}...")

    deeplabcut.analyze_videos(
        dlc_config_path,
        [session['video_path']],
        shuffle=1,
        videotype=".mp4",
        save_as_csv=True,
        cropping=session['cropping_coords'],
        destfolder=session['results_path'],
    )


# Create Grids and save as Grid Files

If DLC tracking was already done separately and crop bounds are already known for each session (as with this dataset), you can skip the "Get DeepLabCut Cropping Bounds" / "Analyze the Videos with DeepLabCut" sections above entirely and use the cell below instead.

(Note: if DLC output lives somewhere other than this project's own `dlc_results\` folder — e.g. a separate external folder structure like this dataset's `DLC_Output\Mouse_X_Trial_Y\` — that needs a one-time reorganize/copy step into `dlc_results\` first, done once outside this notebook rather than as a permanent cell here, since that folder structure is specific to how that particular DLC run was organized.)

In [ ]:
# Synthesize the same boundary-points / cropping-bounds files that
# get_labyrinth_boundary_and_cropping() would have saved from clicking --
# built directly from the X1/X2/Y1/Y2 already known for this dataset.
#
# Note on provenance: get_labyrinth_boundary_and_cropping() computes
# cropping_coords FROM clicked boundary_points, so it correctly saves
# "derived_from_boundary": True. Here the direction is reversed -- we
# already have X1/X2/Y1/Y2 and construct a boundary_points rectangle from
# them, with no human visual verification that the maze actually falls
# within those bounds in the cropped video. Recording that accurately
# (rather than copying the flag value from the real function) matters if
# anyone audits grid alignment later and wants to know which sessions were
# manually verified vs. reconstructed from metadata.
now = datetime.now(); print(now)

print(f"Synthesizing boundary/cropping files for {len(mouseinfo)} sessions...")

for _, row in mouseinfo.iterrows():
    session_name = f"Session{int(row['session']):04d}"
    X1, X2, Y1, Y2 = int(row["x1"]), int(row["x2"]), int(row["y1"]), int(row["y2"])

    boundary_points = np.array([[X1, Y1], [X1, Y2], [X2, Y2], [X2, Y1]])  # TL, BL, BR, TR
    np.save(os.path.join(grid_boundaries_path, f"{session_name}_Boundary_Points.npy"), boundary_points)

    coord_data = {
        "session": session_name, "X1": X1, "X2": X2, "Y1": Y1, "Y2": Y2,
        "width": X2 - X1, "height": Y2 - Y1,
        "boundary_points": boundary_points.tolist(),
        "derived_from_boundary": False,
        "synthesized_from_known_crop_bounds": True,
    }
    np.save(os.path.join(dlc_cropping_bounds_path, f"{session_name}_DLC_Cropping_Bounds.npy"), coord_data)

print("Done.")

Now that boundary/cropping files exist (synthesized above) and DLC output exists in `dlc_results\` under the matching `SessionXXXX` names, `batch_create_grids()` works unmodified.

In [ ]:
now = datetime.now()
NUM_SQUARES = 10

print(f"\nCreating grids for {len(mouseinfo)} sessions (num_squares={NUM_SQUARES})...")

grid_results = dlc_utils.batch_create_grids(
    mouseinfo_df=mouseinfo,
    boundaries_directory=grid_boundaries_path,
    grid_files_directory=grid_path,
    cropping_directory=dlc_cropping_bounds_path,
    num_squares=NUM_SQUARES,
)
print(grid_results)

# Initial Visualizations

### Plot the Scatterplot with Grid overlayed for each trial

In [ ]:
now = datetime.now()

print("\n--- Batch Processing Example ---")
batch_results = dlc_utils.batch_create_grid_scatter_plots(
    mouseinfo_df=mouseinfo,
    dlc_results_directory=dlc_results_path,
    grid_files_directory=grid_path,
    figures_directory=figure_path,
    dlc_scorer=dlc_scorer,
    bodypart='Belly',
    likelihood_threshold=0.6, # rough threshold to visualize when mice are in maze
    figure_size=(3, 3),
    show_plots=False,  # Don't display plots during batch processing
    rotate_180=True,  # matches the same rotation applied for grid_number assignment
)

### Create Trajectory Plots with Grid Overlaid

In [ ]:
now = datetime.now()

print("\n--- Different Colormaps Example ---")
colormaps = ['viridis']

for colormap in colormaps:
    print(f"\nCreating trajectory plots with {colormap} colormap...")
    batch_results = dlc_utils.batch_create_trajectory_plots(
        mouseinfo_df=mouseinfo,
        dlc_results_directory=dlc_results_path,
        grid_files_directory=grid_path,
        figures_directory=figure_path,  # batch_create_trajectory_plots() appends "trajectory_plots" itself -- passing figure_path + "trajectory_plots" here double-nested it
        dlc_scorer=dlc_scorer,
        bodypart='Belly',
        likelihood_threshold=0.6,
        colormap=colormap,
        show_plots=False,
        rotate_180=True,  # matches the same rotation applied for grid_number assignment
    )

# Create CSVs with Grid Numbers

`batch_append_grid_numbers()` also now works unmodified, as long as `bodyparts=["Belly"]` is passed explicitly (this maze's canonical tracked point — the default list uses lowercase names that won't match this data's capitalized DLC bodypart columns).

In [ ]:
now = datetime.now()

# rotate_180=True: this camera's video is rotated 180 degrees relative to
# the maze orientation assumed in constants.py (start=top-left). Rather than
# rotating the videos or rebuilding the topology, we rotate the tracked
# coordinates into alignment before grid assignment -- grids/crop bounds/raw
# DLC tracking are unaffected and don't need to be redone.
grid_number_results = dlc_utils.batch_append_grid_numbers(
    mouseinfo_df=mouseinfo,
    dlc_results_directory=dlc_results_path,
    grid_files_directory=grid_path,
    dlc_scorer=dlc_scorer,
    bodyparts=["Belly"],
    save_directory=dlc_results_path,
    rotate_180=True,
)
print(grid_number_results)